# Week 6 — Building the real job-postings sample

This notebook builds the `data/sample_job_postings.csv` file that the Week 6
lessons and the capstone use. It is the **explained, step-by-step** version of
`scripts/build_job_postings_sample.py` (the headless one-command equivalent).

**Goal:** get ~60 rows of *real* job postings — with full `job_description`
text — without needing a Kaggle login.

**What you'll learn here:**
- how to pull rows from a public dataset through the Hugging Face
  `datasets-server` API (no auth, standard library only),
- how to map a messy upstream schema onto a clean, stable schema,
- how to filter, deduplicate, and write a small reproducible CSV.

> Run the cells top to bottom. The collection step makes a handful of small HTTP
> requests, so you need an internet connection.

## 1. The data source

We use [`xanderios/linkedin-job-postings`](https://huggingface.co/datasets/xanderios/linkedin-job-postings)
on the Hugging Face Hub. It is a **no-login mirror** of the same Kaggle
[LinkedIn Job Postings (2023-2024)](https://www.kaggle.com/datasets/arshkon/linkedin-job-postings)
dataset cited in the Week 6 README.

Why this one?
- it contains a real, free-text **`description`** column (often 1,500–4,000
  characters) — exactly the long text our compression lesson needs,
- it is reachable without Kaggle credentials, so the notebook is reproducible
  for everyone.

Always follow the upstream dataset's license and terms when you reuse it.

## 2. The datasets-server "rows" API

Hugging Face exposes a public read API for hosted datasets:

```
https://datasets-server.huggingface.co/rows?dataset=<id>&config=default&split=train&offset=<n>&length=<n>
```

- **No authentication** for public datasets.
- You page through the data with `offset` and `length` (up to 100 rows/request).
- The response is JSON; each item has a `row` dict with the actual fields.

We only need Python's standard library (`urllib`, `json`, `csv`) — no `pandas`,
no `requests`, no Kaggle CLI.

In [ ]:
import csv
import datetime as dt
import json
import urllib.parse
import urllib.request
from pathlib import Path

# A no-login mirror of the Kaggle "LinkedIn Job Postings (2023-2024)" dataset.
DATASET = "xanderios/linkedin-job-postings"
ROWS_API = "https://datasets-server.huggingface.co/rows"

# Target schema shared by the Week 6 lessons and the capstone template.
SCHEMA = [
    "job_id", "job_title", "company", "location",
    "job_description", "job_skills", "posted_date", "source",
]

# Keep tech/data roles so the "skill analyzer / learning path" theme is relevant.
TITLE_KEYWORDS = (
    "data", "engineer", "analyst", "developer", "scientist",
    "machine learning", "ml ", "software", "devops", "ai ", "ai/", "programmer",
)

TARGET_ROWS = 60
PAGE_SIZE = 100
MAX_OFFSET = 4000           # scan budget; tech roles are only a fraction of all rows
DESCRIPTION_MAX_CHARS = 1500

print("Config ready. Target:", TARGET_ROWS, "rows from", DATASET)

## 3. Fetch one page and inspect the raw structure

Before mapping anything, let's see what a row actually looks like. `fetch_page`
wraps one API call and returns a list of `row` dicts.

In [ ]:
def fetch_page(offset: int, length: int) -> list:
    """Fetch one page of rows from the Hugging Face datasets-server API."""
    params = urllib.parse.urlencode({
        "dataset": DATASET, "config": "default", "split": "train",
        "offset": offset, "length": length,
    })
    url = f"{ROWS_API}?{params}"
    req = urllib.request.Request(url, headers={"User-Agent": "week06-sample-builder"})
    with urllib.request.urlopen(req, timeout=60) as resp:
        payload = json.load(resp)
    return [item["row"] for item in payload.get("rows", [])]


# Demonstration: peek at the first few rows.
peek = fetch_page(offset=0, length=3)
print("rows returned:", len(peek))
print("available columns:\n ", ", ".join(peek[0].keys()))

print("\nfirst row sample:")
print("  title:      ", peek[0]["title"])
print("  location:   ", peek[0]["location"])
print("  work_type:  ", peek[0]["work_type"])
print("  skills_desc:", repr(peek[0]["skills_desc"]))
print("  listed_time:", peek[0]["listed_time"], "(epoch milliseconds)")
print("  description:", len(peek[0]["description"]), "chars")

## 4. Map the upstream columns onto our schema

The raw dataset has 28 columns with LinkedIn-specific names. We only want the
eight columns the rest of Week 6 expects, so we map them:

| Our column | Source field | Notes |
|---|---|---|
| `job_id` | `job_id` | |
| `job_title` | `title` | |
| `company` | — | left empty: this mirror only ships a numeric `company_id`, not a name |
| `location` | `location` | |
| `job_description` | `description` | truncated to ~1,500 chars to keep the file small |
| `job_skills` | `skills_desc` | usually empty upstream → analyzer infers skills from text |
| `posted_date` | `listed_time` | epoch **milliseconds** → `YYYY-MM-DD` |
| `source` | — | constant tag for provenance |

The helpers below also **clean whitespace**, **drop rows missing a title or
description**, and **keep only tech/data roles**.

In [ ]:
def looks_techy(title: str) -> bool:
    low = f" {title.lower()} "
    return any(kw in low for kw in TITLE_KEYWORDS)


def epoch_ms_to_date(value) -> str:
    try:
        ms = float(value)
    except (TypeError, ValueError):
        return ""
    return dt.datetime.fromtimestamp(ms / 1000.0, dt.timezone.utc).strftime("%Y-%m-%d")


def clean_text(value) -> str:
    if value is None:
        return ""
    # Collapse newlines/whitespace so each posting stays in one tidy CSV cell.
    return " ".join(str(value).replace("\r\n", "\n").split())


def map_row(row: dict):
    """Map one upstream row to our schema, or return None to skip it."""
    title = clean_text(row.get("title"))
    description = clean_text(row.get("description"))
    if not title or not description:
        return None                       # skip rows missing the essentials
    if not looks_techy(title):
        return None                       # keep only tech/data roles
    if len(description) > DESCRIPTION_MAX_CHARS:
        description = description[: DESCRIPTION_MAX_CHARS - 1].rstrip() + "\u2026"
    return {
        "job_id": row.get("job_id", ""),
        "job_title": title,
        "company": "",                     # mirror has company_id only, not a name
        "location": clean_text(row.get("location")),
        "job_description": description,
        "job_skills": clean_text(row.get("skills_desc")),
        "posted_date": epoch_ms_to_date(row.get("listed_time")),
        "source": "linkedin_kaggle_2023_2024",
    }


# Demonstration: map the peeked rows and show one full mapped record.
mapped_demo = [m for m in (map_row(r) for r in peek) if m]
print(f"{len(mapped_demo)} of {len(peek)} peeked rows kept after mapping/filtering")
if mapped_demo:
    preview = dict(mapped_demo[0])
    preview["job_description"] = preview["job_description"][:160] + " ..."
    print(json.dumps(preview, indent=2))

## 5. See the tech/data filter in action

Job titles are noisy. `looks_techy` keeps a title if it contains any of our
keywords. Here is which titles pass on a larger page — handy for sanity-checking
the keyword list before a full run.

In [ ]:
for title in (clean_text(r.get("title")) for r in fetch_page(0, 25)):
    mark = "KEEP" if looks_techy(title) else "skip"
    print(f"  [{mark}] {title[:60]}")

## 6. Collect the full sample

Now we page through the dataset until we have `TARGET_ROWS` tech/data postings.
We **deduplicate** by `job_id` and stop early once the target is reached or we
hit the scan budget (`MAX_OFFSET`).

In [ ]:
collected = []
seen_ids = set()
offset = 0

while len(collected) < TARGET_ROWS and offset < MAX_OFFSET:
    rows = fetch_page(offset, PAGE_SIZE)
    if not rows:
        break
    for row in rows:
        mapped = map_row(row)
        if mapped is None or mapped["job_id"] in seen_ids:
            continue
        seen_ids.add(mapped["job_id"])
        collected.append(mapped)
        if len(collected) >= TARGET_ROWS:
            break
    offset += PAGE_SIZE
    print(f"  scanned through offset {offset:>4} -> collected {len(collected)} rows")

print(f"\nDone. Collected {len(collected)} tech/data postings.")

## 7. Write the CSV

Jupyter runs with the notebook's own folder as the working directory. This
notebook lives in `week_06/scripts/`, so the data folder is `../data`.

In [ ]:
data_dir = Path.cwd().parent / "data" if Path.cwd().name == "scripts" else Path.cwd() / "data"
if not data_dir.is_dir():
    raise FileNotFoundError(f"Could not find the Week 6 data directory at {data_dir}")

out_path = data_dir / "sample_job_postings.csv"
with out_path.open("w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=SCHEMA)
    writer.writeheader()
    writer.writerows(collected)

print(f"Wrote {len(collected)} rows to {out_path}")

## 8. Verify the output

A quick read-back confirms the row count, the columns, how populated each column
is, and a few example titles. Note `company` is empty and `job_skills` is sparse
— expected limitations of this mirror (documented in `data/README.md`).

In [ ]:
rows = list(csv.DictReader(out_path.open(encoding="utf-8")))
print("row count:", len(rows))
print("columns:  ", list(rows[0].keys()))

print("\nnon-empty count per column:")
for col in rows[0]:
    filled = sum(1 for r in rows if r[col].strip())
    print(f"  {col:16} {filled:>3}")

print("\nsample titles + description lengths:")
for r in rows[:8]:
    print(f"  {r['posted_date']} | {len(r['job_description']):>4} chars | {r['job_title'][:50]}")

## 9. Limitations & how to extend

- **`company` is empty** and **`job_skills` is sparse** in this mirror. The
  analyzer is expected to infer skills from `job_description`. For populated
  company names you'd need the original Kaggle release (separate `companies.csv`)
  plus Kaggle credentials.
- **Descriptions are truncated** to ~1,500 chars to keep the classroom file
  small. Raise `DESCRIPTION_MAX_CHARS` for richer text.
- **Want more rows or a different focus?** Increase `TARGET_ROWS`, widen
  `MAX_OFFSET`, or edit `TITLE_KEYWORDS` (e.g. add `"product"`, `"security"`).
- **Headless equivalent:** running `python scripts/build_job_postings_sample.py`
  performs these same steps and writes the same file from the command line.